#**準備**

Googleドライブのマウントとデータのダウンロード


In [ ]:
import os, sys, shutil
os.environ['TRANSFORMERS_CACHE'] = '/content/cache/hf'
os.environ['HF_HOME']            = '/content/cache/hf'
os.environ['HF_DATASETS_CACHE']  = '/content/cache/hf/datasets'
os.environ['TORCH_HOME']         = '/content/cache/torch'
os.environ['TORCH_CACHE']        = '/content/cache/torch'
os.environ['XDG_CACHE_HOME']     = '/content/cache'
os.makedirs('/content/cache', exist_ok=True)

# @title
# Google Driveをマウント
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title
# データのダウンロード
! rm -r -f SpeakerClassify
! git clone https://github.com/akio-kobayashi/SpeakerClassify.git
! pip -q install lightning einops scikit-learn
! pip -q install torch-summary
#%cd /content/drive/MyDrive/話者識別/
#! tar zxf SpeechData.tar.gz
%cd /content/SpeakerClassify

#**実験のセットアップ**

話者識別モデルの学習と評価に必要な情報の入力

In [ ]:
#学習器
student_id='00000001'#@param{type:"string"}
# ダウンロードしたデータが格納されるフォルダ（変更しない）
rootdir='/content/drive/Shareddrives/専門演習I/後期後半/SpeechData'#@param{type:'string'}
# 作ったモデルや評価結果を保存するフォルダ（変更しない）
save_root='/content/drive/Shareddrives/専門演習I/後期後半/SpeakerModel/'#@param {type:"string"}
save_dir=os.path.join(save_root,student_id)
# モデルの名前（新しいモデルを作るたびに変更すること）
model_name='baseline'#@param {type:"string"}
# 学習に使うデータ（録音データ一覧，CSVフォーマット，変更しない）
csv_path='data.csv'
# 話者識別の結果を記録するファイル
report_csv='report.csv'
detail_csv='detail.csv'
# 混同行列（画像）を記録するファイル
confusion_matrix_fig='confusion.png'
# 学習したモデル（変更しない）
ckpt="last.ckpt"

# モデルパラメータ
max_epochs = 10#@param {type:"integer"}
# 認識する話者の人数（専門演習に参加している学生の数）
num_speakers=13#@param {type:"integer"}
# 学習率
lr=1.e-3#@param{type:"number"}
if not student_id:
  raise ValueError('wrong student id')
check_dir=os.path.join(save_dir, model_name)
if "SpeechData" in check_dir:
  raise ValueError('wrong directory name')

if os.path.exists(check_dir):
  print(check_dir)
  shutil.rmtree(check_dir)

# コンフィギュレーションファイルの読み込み
import yaml
with open('config.yaml') as f:
    config = yaml.safe_load(f)
config['speakers']['save_path'] = os.path.join(save_dir, config['speakers']['save_path'])
if not os.path.exists(save_dir):
  os.makedirs(save_dir)
config['optimizer']['lr']=lr
config['trainer']['max_epochs'] = max_epochs
config['num_speakers'] = num_speakers
config['csv'] = os.path.join(rootdir, csv_path)
config['logger']['save_dir'] = save_dir
config['logger']['name'] = model_name
config['report']['path'] = os.path.join(save_dir, model_name, "version_"+str(config['logger']['version']), report_csv)
config['report']['detail'] = os.path.join(save_dir, model_name, "version_"+str(config['logger']['version']), detail_csv)
config['report']['confusion_matrix'] = os.path.join(save_dir, model_name, "version_"+str(config['logger']['version']), confusion_matrix_fig)
config['checkpoint_path']=os.path.join(save_dir, model_name, "version_"+str(config['logger']['version']), 'checkpoints', ckpt)

#**モデルの学習**

##**モデルの準備**

ベースラインモデルのコードなので，これに改良を行う

In [ ]:
import torch
from torch import nn
from typing import List, Tuple

# ブロック単位の定義
class MyBlock(nn.Module):
    """
    MyBlockは畳み込み層，正規化，活性化関数を一つにまとめたモジュールである
    ・引数で正規化方法や活性化関数を指定可能
    ・オプションで残差接続を有効にすることも可能
    """
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding=0,
                 activation="ReLU", norm_type="BatchNorm2d", use_residual=False, dropout_rate=0.0):
        super().__init__()
        self.use_residual = use_residual
        # 畳み込み層
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, bias=False)

        # 正規化層（選択可能）
        if norm_type == "BatchNorm2d":
            self.norm = nn.BatchNorm2d(out_channels)
        elif norm_type == "LayerNorm":
            self.norm = nn.LayerNorm([out_channels, 1, 1])  # 入力形状に応じて調整
        elif norm_type == "InstanceNorm2d":
            self.norm = nn.InstanceNorm2d(out_channels)
        else:
            raise ValueError(f"Unsupported normalization type: {norm_type}")

        # 活性化関数（選択可能）
        if activation == "ReLU":
          self.activation = nn.ReLU(inplace=False)
        else:
          self.activation = getattr(nn, activation)()

        self.dropout = nn.Dropout2d(p=dropout_rate) if dropout_rate > 0 else nn.Identity()

        # 残差接続
        if self.use_residual:
            if in_channels != out_channels or stride != (1, 1):
                self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False)
            else:
                self.shortcut = nn.Identity()

    def forward(self, x):
        residual=x
        # 畳み込み → 正規化 → 活性化
        y = self.conv(x)
        y = self.norm(y)
        y = self.activation(y)
        y = self.dropout(y)

        if self.use_residual:
          shortcut = self.shortcut(residual)
          if shortcut.shape != y.shape:
            shortcut = torch.nn.functional.interpolate(shortcut, size=y.shape[2:], mode="nearest")
          y = y+shortcut

        return y


class MyModel(nn.Module):
    """
    MyModelは話者識別用のニューラルネットワークモデルである
    ・複数のMyBlockを積み重ねて特徴抽出を行う
    ・特徴抽出後，全結合層で話者分類を行う
    ・block_configsを引数に渡すことで，構造を柔軟に変更可能
    """
    def __init__(self, num_speakers: int, block_configs: List[Tuple[int, int, Tuple[int, int], Tuple[int, int], Tuple[int, int]]]):
        super().__init__()

        # 畳み込みブロックをリストで管理
        self.blocks = nn.ModuleList([
            MyBlock(in_channels, out_channels, kernel_size, stride, padding,
                    activation="ReLU", norm_type="BatchNorm2d")
            for in_channels, out_channels, kernel_size, stride, padding in block_configs
        ])

        # 最後のブロックの出力チャネル数を取得
        final_output_channels = block_configs[-1][1]

        # 全結合層（ボトルネック層 → 分類層）
        bottleneck_channels = final_output_channels // 2
        self.feedforward = nn.Sequential(
            nn.Linear(final_output_channels, bottleneck_channels),  # ボトルネック
            nn.ReLU(),  # 活性化関数
            nn.Linear(bottleneck_channels, num_speakers)  # 話者数に対応する分類層
        )

    def forward(self, x):
        # 畳み込みブロックを順番に適用
        for block in self.blocks:
            x = block(x)

        # グローバル平均プーリング（特徴マップを1次元に集約）
        x = torch.mean(torch.mean(x, dim=-1), dim=-1)

        # 全結合層で最終分類
        return self.feedforward(x)


# サンプルのブロック設定（入力チャンネル数，出力チャンネル数，カーネルサイズ，ストライド，パディング）
# モデル構造の変更に用いる
block_configs = [
    (1, 64, (16, 16), (2, 2), (8, 8)),   # 第1ブロック
    (64, 128, (8, 8), (2, 2), (4, 4)),  # 第2ブロック
    (128, 256, (4, 4), (2, 2), (2, 2)), # 第3ブロック
    (256, 512, (4, 4), (1, 1), (2, 2))  # 第4ブロック
]

##**モデルの動作チェック**

In [ ]:
model = MyModel(num_speakers, block_configs)
# ダミーのデータを準備
input_tensor=torch.rand(1, 1, 80, 8192)
output_tensor = model(input_tensor)
# 以下を出力できればモデルは正しく動作する
print(output_tensor.shape)

from torchsummary import summary
summary(model, [(1, 80, 8192)])

##**モデルの学習を実行する**

In [ ]:
# @title
import train
import solver
from speech_dataset import SpeechDataset
from speech_dataset import data_processing
import torch.utils.data as data
import pytorch_lightning as pl
from lightning.pytorch.loggers import TensorBoardLogger
from pytorch_lightning.utilities.model_summary import ModelSummary

train.set_seed(42)
lite = solver.LightningSolver(config, model.cuda())
train_dataset = SpeechDataset(csv_path=config['csv'], save_path=config['speakers']['save_path'], valid=False)
train_loader = data.DataLoader(dataset=train_dataset,
                                   batch_size=config['batch_size'],
                                   num_workers=1,
                                   pin_memory=True,
                                   shuffle=True,
                                   collate_fn=lambda x: data_processing(x))
valid_dataset = SpeechDataset(csv_path=config['csv'], speaker2idx=train_dataset._speaker2idx(), valid=True)
valid_loader = data.DataLoader(dataset=valid_dataset,
                                   batch_size=config['batch_size'],
                                   num_workers=1,
                                   pin_memory=True,
                                   shuffle=False,
                                   collate_fn=lambda x: data_processing(x))

callbacks = [
        pl.callbacks.ModelCheckpoint( **config['checkpoint'])
]
logger = TensorBoardLogger(**config['logger'])

if 'profiler' in config['trainer']:
  config['trainer'].pop('profiler')
trainer = pl.Trainer( callbacks=callbacks,
                          logger=logger,
                          **config['trainer'] )
trainer.fit(model=lite, train_dataloaders=train_loader, val_dataloaders=valid_loader)

#**（学習に用いていない）未知のデータを評価する**

In [ ]:
# @title
import predict
predict.predict(config, model, data_type="eval")